In [10]:
!pip install pypdf

In [8]:
import boto3, json, time
import tempfile
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain.vectorstores import OpenSearchVectorSearch
from langchain.chains import RetrievalQA
from langchain_aws import ChatBedrock, ChatBedrockConverse
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.chains import RetrievalQA
from langchain_aws import BedrockEmbeddings
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader

In [2]:
def get_data_from_s3(bucket_name, key):
    s3 = boto3.client(
        's3',
        region_name=region,
    )
    response = s3.get_object(Bucket=bucket_name, Key=key)
    data = response['Body'].read().decode('utf-8')

    return data

In [6]:
def get_pdf_docs_from_s3(bucket_name, key, region="us-east-1"):
    # Create an S3 client
    s3 = boto3.client('s3', region_name=region)
    
    # Get the PDF bytes from S3
    response = s3.get_object(Bucket=bucket_name, Key=key)
    pdf_bytes = response['Body'].read()
    
    # Write PDF bytes into a temporary file
    with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp_file:
        tmp_file.write(pdf_bytes)
        tmp_file.flush()  # Ensure data is written to disk
        
        # Pass the path of the temporary file to PyPDFLoader
        loader = PyPDFLoader(tmp_file.name)
        documents = loader.load()
    
    return documents

In [3]:
session = boto3.session.Session()
region = session.region_name
credentials = session.get_credentials()
awsauth = AWSV4SignerAuth(credentials, region, service='aoss')
aoss_client = session.client('opensearchserverless')

suffix = "demo"
bucket_name = "bucket-test-cj"
vector_store_name = f"bedrock-sample-rag-{suffix}"
index_name = f"bedrock-sample-index-{suffix}"

In [11]:
s3_data = get_pdf_docs_from_s3("bucket-test-cj", "BILL-Q2-25-Press-Release-2-6-25.pdf")

In [12]:
type(s3_data)

list

In [14]:
len(s3_data)

14

In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)

chunks = text_splitter.split_documents(s3_data)
len(chunks)

41

In [17]:
embeddings = BedrockEmbeddings(model_id="amazon.titan-embed-text-v2:0")
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)

In [18]:
llm = ChatBedrockConverse(
    model="anthropic.claude-3-sonnet-20240229-v1:0",
    temperature=0.0,
    top_p=0.9,
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_kwargs={"k": 3})
)

In [19]:
qa_chain('How was Q2 of 2025 for BILL?')

/tmp/ipykernel_28570/4161104552.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  qa_chain('How was Q2 of 2025 for BILL?')


{'query': 'How was Q2 of 2025 for BILL?',
 'result': 'Based on the financial results shared in the press release, Q2 of fiscal year 2025 was a strong quarter for BILL:\n\n- Total revenue increased 14% year-over-year to $362.6 million.\n- Core revenue (subscription and transaction fees) increased 16% year-over-year to $319.6 million.\n- Subscription revenue grew 7% year-over-year to $67.7 million.\n- Transaction fee revenue grew 19% year-over-year to $251.9 million.\n\nThe CEO René Lacerte highlighted that they delivered strong financial results and continued innovating rapidly to execute on their vision of being the leading financial operations platform for small and mid-sized businesses. The CFO John Rettig also stated they expanded their non-GAAP operating margin and continued executing well across the company in Q2.\n\nOverall, the double-digit revenue growth, strong performance in their core subscription and transaction businesses, and positive commentary from the executives indica